# Data Cleaning

## Import & Load Data

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("raw/aug_personal_transactions_with_UserId.csv")

# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10806 entries, 0 to 10805
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   User ID           806 non-null    float64
 1   Date              10806 non-null  str    
 2   Description       10806 non-null  str    
 3   Amount            10806 non-null  float64
 4   Transaction Type  10806 non-null  str    
 5   Category          10806 non-null  str    
 6   Account Name      10806 non-null  str    
dtypes: float64(2), str(5)
memory usage: 591.1 KB


## Basic Cleaning

In [4]:
# Remove User ID column if it exists
df = df.drop(columns=["User ID"], errors="ignore")

# Datetime formatting
df["Date"] = pd.to_datetime(df["Date"])
df["Day of Week"] = df["Date"].dt.dayofweek
df["Month"] = df["Date"].dt.month

df["Amount"] = pd.to_numeric(df["Amount"])

for col in ["Description", "Transaction Type", "Category", "Account Name"]:
    df[col] = df[col].astype(str).str.strip()

In [5]:
df.head()

,Date,Description,Amount,Transaction Type,Category,Account Name,Day of Week,Month
0,2018-01-01,Amazon,11.11,debit,Shopping,Platinum Card,0,1
1,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking,1,1
2,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,Silver Card,1,1
3,2018-01-03,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2,1
4,2018-01-04,Netflix,11.76,debit,Movies & DVDs,Platinum Card,3,1


## Defining Target Categories

In [6]:
# Counting unique descriptions 
desc_counts = df["Description"].value_counts().reset_index()
desc_counts.columns = ["Description", "Count"]
pd.set_option("display.max_rows", None)

display(desc_counts)

,Description,Count
0,Credit Card Payment,1848
1,Grocery Store,1417
2,Biweekly Paycheck,608
3,Amazon,606
4,Gas Company,435
5,American Tavern,433
6,Hardware Store,405
7,Starbucks,398
8,Brewing Company,353
9,City Water Charges,295


In [7]:
# Organizing target words into 5 categories using unique descriptions
def category(description):
    desc = str(description).lower()

    if (
        "paycheck" in desc
        or "construction" in desc
    ):
        return "Income"

    elif (
        "mortgage" in desc
        or "credit card payment" in desc
        or "water" in desc
        or "phone" in desc
        or "internet" in desc
        or "power" in desc
        or "gas company" in desc
        or "state farm" in desc
    ):
        return "Bills & Payments"

    elif (
        "shell" in desc
        or "valero" in desc
        or "circle k" in desc
        or "chevron" in desc
        or "gas station" in desc
        or "go mart" in desc
    ):
        return "Transportation"

    elif (
        "grocery" in desc
        or "restaurant" in desc
        or "starbucks" in desc
        or "tavern" in desc
        or "brewing" in desc
        or "brunch" in desc
        or "chick-fil-a" in desc
        or "pizza" in desc
        or "diner" in desc
        or "deli" in desc
        or "bbq" in desc
        or "bakery" in desc
        or "food truck" in desc
        or "market" in desc
        or "steakhouse" in desc
        or "grill" in desc
        or "pub" in desc
    ):
        return "Food & Dining"

    else:
        return "Shopping, Personal & Entertainment"


df["Category"] = df["Description"].apply(category)

print(df["Category"].value_counts())

Category
Bills & Payments                      3820
Food & Dining                         3725
Shopping, Personal & Entertainment    2330
Income                                 673
Transportation                         258
Name: count, dtype: int64


## Model Features

In [8]:
# Binary encode transaction type (credit = 0, debit = 1)
df["Transaction Type"] = df["Transaction Type"].str.lower().map({
    "credit": 0,
    "debit": 1
})

df["Description_grouped"] = df["Description"]

# One-hot encode description and account name
desc_ohe = pd.get_dummies(df["Description_grouped"], prefix="Desc", dtype=int)
acct_ohe = pd.get_dummies(df["Account Name"], prefix="Acct", dtype=int)

df["Amount_scaled"] = StandardScaler().fit_transform(df[["Amount"]])

# Label encode category target
df["Category"] = df["Category"].astype("category")
df["Category_Label"] = df["Category"].cat.codes

# Show category labels
df[["Category", "Category_Label"]].drop_duplicates().sort_values("Category_Label").reset_index(drop=True)

,Category,Category_Label
0,Bills & Payments,0
1,Food & Dining,1
2,Income,2
3,"Shopping, Personal & Entertainment",3
4,Transportation,4


## Cleaned Data

In [9]:
final_df = pd.concat(
    [
        df[
            [
                "Date",
                "Amount",
                "Transaction Type",
                "Category",
                "Day of Week",
                "Month",
                "Description_grouped",
                "Category_Label",
                "Amount_scaled",
            ]
        ].reset_index(drop=True),
        desc_ohe.reset_index(drop=True),
        acct_ohe.reset_index(drop=True),
    ],
    axis=1,
)

final_df.to_csv("cleaned/transactions_cleaned.csv", index=False)

final_df.head()

,Date,Amount,Transaction Type,Category,Day of Week,Month,Description_grouped,Category_Label,Amount_scaled,Desc_Amazon,...,Desc_Sushi Restaurant,Desc_Target,Desc_Thai Restaurant,Desc_Tiny Deli,Desc_Valero,Desc_Vietnamese Restaurant,Desc_Wendy's,Acct_Checking,Acct_Platinum Card,Acct_Silver Card
0,2018-01-01,11.11,1,"Shopping, Personal & Entertainment",0,1,Amazon,3,-0.520732,1,...,0,0,0,0,0,0,0,0,1,0
1,2018-01-02,1247.44,1,Bills & Payments,1,1,Mortgage Payment,0,5.791472,0,...,0,0,0,0,0,0,0,1,0,0
2,2018-01-02,24.22,1,Food & Dining,1,1,Thai Restaurant,1,-0.453797,0,...,0,0,1,0,0,0,0,0,0,1
3,2018-01-03,2298.09,0,Bills & Payments,2,1,Credit Card Payment,0,11.155669,0,...,0,0,0,0,0,0,0,0,1,0
4,2018-01-04,11.76,1,"Shopping, Personal & Entertainment",3,1,Netflix,3,-0.517413,0,...,0,0,0,0,0,0,0,0,1,0
